# Task 4 - Step 2: Freeze the models, extract outputs, fit Mahalanobis, calibrate thresholds

**No CIFAR-100 image is used in this notebook.** Everything that defines a detector is fixed here:

1. Penultimate features f(x) and logits z(x) of every frozen model for CIFAR-10 **train (unaugmented), validation
   and test**.
2. Closed-set accuracy (CSA) on the full CIFAR-10 test set, from the ten known-class logits only.
3. Mahalanobis fitted on the **unaugmented CIFAR-10 training features** of the Vanilla model: class means and one
   shared diagonal covariance with 1e-6 added.
4. Rejection thresholds: for every (model, score), τ = 95th percentile of unknownness on CIFAR-10 **validation**.
5. A freeze manifest (SHA-256 of checkpoints, configs, outputs, Mahalanobis state and thresholds). Notebook 03 can
   load unknowns only if all of these are unchanged.

In [1]:
# ---- Task 4 common header (identical in every Task 4 notebook) ----
# NOTE: no CIFAR-100 image is decoded or loaded anywhere except notebook 03, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "task4" / "training.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.config import load_config
from task4.data import cifar10
from task4.models.resnet_cifar import build_model

T4 = REPO / "task4"
CFG_DIR = T4 / "configs"
SEED = 6304
# Smoke mode (env TASK4_SMOKE=1): 2 epochs on 2000 training images; outputs under _smoke/; notebook 03 uses
# RANDOM stand-in "unknown" images, so no CIFAR-100 image is touched.
SMOKE = os.environ.get("TASK4_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T4 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T4 / "results" / "tables"       # data-preparation files (same path in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T4 / "checkpoints" / SUB            # git-ignored
CACHE = T4 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUNS = ["vanilla", "gcsc", "proser"]
LABEL = {"vanilla": "Vanilla", "gcsc": "GCSC", "proser": "PROSER"}
SCORES = ["msp", "mls", "energy", "mahalanobis"]         # post-hoc scores on the frozen Vanilla model
SCORE_LABEL = {"msp": "MSP", "mls": "MLS", "energy": "Energy", "mahalanobis": "Mahalanobis",
               "placeholder": "PROSER placeholder"}


def run_dir(n):
    return CKPT / load_config(CFG_DIR, n)["run_name"]


def load_trained(n):
    """Frozen model with the selected checkpoint of run ``n``."""
    cfg = load_config(CFG_DIR, n)
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(cfg["model"]["num_classes"], cfg["model"]["dummy_classifiers"], cfg["seed"])
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck, cfg


@torch.no_grad()
def extract(model, x_u8, batch_size=512):
    """Penultimate features and logits of a frozen model (no augmentation)."""
    feats, logits = [], []
    for s in range(0, len(x_u8), batch_size):
        f, z = model(cifar10.normalize(x_u8[s:s + batch_size].to(DEVICE)))
        feats.append(f.float().cpu()); logits.append(z.float().cpu())
    return torch.cat(feats), torch.cat(logits)


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
from task4.evaluation.metrics import validation_threshold
from task4.scores import MahalanobisScore, POST_HOC

known = {p: cifar10.load(p) for p in ("train", "val", "test")}
if SMOKE:
    known = {k: (v[0][:2000], v[1][:2000]) for k, v in known.items()}
OUT, CSA = {}, {}
for n in RUNS:
    model, ck, cfg = load_trained(n)
    K = cfg["model"]["num_classes"]
    OUT[n] = {p: dict(zip(("feat", "logits"), extract(model, known[p][0]))) for p in known}
    for p in known:
        OUT[n][p]["labels"] = known[p][1]
    CSA[n] = float((OUT[n]["test"]["logits"][:, :K].argmax(1) == known["test"][1]).float().mean())
    print(f"{n:8s} selected epoch {ck['epoch']:3d} | val acc {ck['val_acc']:.4f} | CIFAR-10 test CSA {CSA[n]:.4f}")
    del model
torch.save({"outputs": OUT, "csa": CSA}, CACHE / "known_outputs.pt")

vanilla  selected epoch  95 | val acc 0.9526 | CIFAR-10 test CSA 0.9461


gcsc     selected epoch  99 | val acc 0.9568 | CIFAR-10 test CSA 0.9543


proser   selected epoch   1 | val acc 0.9486 | CIFAR-10 test CSA 0.9440


## Mahalanobis (Vanilla training features) and the four post-hoc scores

In [3]:
K = 10
mah = MahalanobisScore().fit(OUT["vanilla"]["train"]["feat"], OUT["vanilla"]["train"]["labels"], K)
torch.save(mah.state(), CACHE / "mahalanobis_vanilla.pt")


def known_scores(run, part):
    """Unknownness of every score for a model/part, from the saved logits/features."""
    z, f = OUT[run][part]["logits"][:, :K], OUT[run][part]["feat"]
    s = {name: fn(z).numpy() for name, fn in POST_HOC.items()}
    if run == "vanilla":
        s["mahalanobis"] = mah(f).numpy()
    if run == "proser":
        dummy = OUT[run][part]["logits"][:, K:].max(1).values
        s["placeholder"] = (dummy - z.max(1).values).numpy()      # strongest dummy vs strongest known
    return s


SCORES_KNOWN = {n: {p: known_scores(n, p) for p in known} for n in RUNS}
rows = []
for n in RUNS:
    for s, u in SCORES_KNOWN[n]["val"].items():
        rows.append({"config": n, "model": LABEL[n], "score": SCORE_LABEL[s], "score_key": s,
                     "threshold_tau": validation_threshold(u, load_config(CFG_DIR, n)["evaluation"]["threshold_percentile"]),
                     "val_accept_rate_at_tau": float((u <= validation_threshold(u)).mean()),
                     "val_mean": float(u.mean()), "val_std": float(u.std())})
THRESH = pd.DataFrame(rows)
THRESH.to_csv(TAB / "task4_thresholds.csv", index=False)
torch.save({"scores_known": SCORES_KNOWN, "thresholds": THRESH.to_dict("records"), "csa": CSA}, CACHE / "known_scores.pt")
THRESH.round(4)

,config,model,score,score_key,threshold_tau,val_accept_rate_at_tau,val_mean,val_std
0,vanilla,Vanilla,MSP,msp,0.1535,0.95,0.0213,0.0790
1,vanilla,Vanilla,MLS,mls,-5.9302,0.95,-9.1706,1.5707
2,vanilla,Vanilla,Energy,energy,-6.0661,0.95,-9.1966,1.5073
3,vanilla,Vanilla,Mahalanobis,mahalanobis,2951.2963,0.95,785.4164,884.1384
4,gcsc,GCSC,MSP,msp,0.1978,0.95,0.0250,0.0851
5,gcsc,GCSC,MLS,mls,-6.1411,0.95,-9.4869,1.8476
6,gcsc,GCSC,Energy,energy,-6.3021,0.95,-9.5176,1.7872
7,proser,PROSER,MSP,msp,0.1944,0.95,0.0277,0.0878
8,proser,PROSER,MLS,mls,-4.3880,0.95,-7.3680,1.4631
9,proser,PROSER,Energy,energy,-4.5917,0.95,-7.4019,1.3884


In [4]:
csa_tbl = pd.DataFrame([{"config": n, "model": LABEL[n], "cifar10_test_csa": CSA[n],
                         "val_acc_selected": json.loads((run_dir(n) / "summary.json").read_text())["best_val_acc"]} for n in RUNS])
csa_tbl.to_csv(TAB / "task4_closed_set_accuracy.csv", index=False)
csa_tbl.round(4)

,config,model,cifar10_test_csa,val_acc_selected
0,vanilla,Vanilla,0.9461,0.9526
1,gcsc,GCSC,0.9543,0.9568
2,proser,PROSER,0.9440,0.9486


## Freeze manifest

In [5]:
from task4.data.cifar10 import file_sha256
from shared.pacs import write_freeze_manifest

frozen = [cifar10.SPLIT_PATH, CACHE / "known_outputs.pt", CACHE / "known_scores.pt", CACHE / "mahalanobis_vanilla.pt",
          TAB / "task4_thresholds.csv"] + sorted(CFG_DIR.glob("*.yaml"))
for n in RUNS:
    frozen += [run_dir(n) / f for f in ("best.pt", "config.yaml", "summary.json")]
manifest = write_freeze_manifest(CACHE / "freeze_manifest.json", frozen,
                                 note="Task 4: models trained and selected on CIFAR-10 validation; scores defined; thresholds calibrated on validation; no CIFAR-100 image used")
print("frozen files:", len(manifest["files"]), "|", manifest["created"])

frozen files: 18 | 2026-09-24 01:15:54
